# Logistic Regression — Adam

Assignment 2: Breast Cancer Wisconsin binary classification. **Target:** 0 = malignant, 1 = benign. Logistic regression and optimizer implemented from scratch in NumPy; scikit-learn only for splitting, scaling, metrics. Train/test split is stratified and scaler is fit only on training data.

## 1. Load dataset and define logistic regression

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
df = pd.read_csv('breast_cancer_dataset.csv')
print('Dataset shape:', df.shape)
print('Target distribution (0=malignant, 1=benign):')
print(df['target'].value_counts().sort_index())
display(df.head())
X = df.drop(columns=['target']).to_numpy(dtype=float)
y = df['target'].to_numpy(dtype=float)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)
X_train = np.column_stack([np.ones(len(X_train_scaled)), X_train_scaled])
X_test = np.column_stack([np.ones(len(X_test_scaled)), X_test_scaled])
print('Train:', len(y_train), 'Test:', len(y_test))

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def predict_proba(X, w):
    return sigmoid(X @ w)

def binary_cross_entropy(X, y, w):
    # Stable logistic loss: log(1+exp(z)) - y*z
    z = X @ w
    return np.mean(np.logaddexp(0, z) - y * z)

def gradient(X, y, w):
    return X.T @ (predict_proba(X, w) - y) / len(y)

def evaluate(X, y, w):
    pred = (predict_proba(X, w) >= 0.5).astype(int)
    return dict(Loss=binary_cross_entropy(X,y,w),
                Accuracy=accuracy_score(y,pred),
                Precision=precision_score(y,pred,zero_division=0),
                Recall=recall_score(y,pred,zero_division=0),
                F1=f1_score(y,pred,zero_division=0))


## 2. Implement optimizer and train

In [ ]:
def train(X, y, epochs=220, lr=0.005, batch_size=32,
          beta1=0.9, beta2=0.999, eps=1e-8, seed=42):
    w = np.zeros(X.shape[1]); m = np.zeros_like(w); v = np.zeros_like(w)
    history = []; step = 0; rng = np.random.default_rng(seed)
    for epoch in range(epochs):
        indices = rng.permutation(len(y))
        for start in range(0, len(y), batch_size):
            idx = indices[start:start+batch_size]
            g = gradient(X[idx], y[idx], w)
            step += 1
            m = beta1*m + (1-beta1)*g
            v = beta2*v + (1-beta2)*(g*g)
            m_hat = m / (1-beta1**step)
            v_hat = v / (1-beta2**step)
            w -= lr * m_hat / (np.sqrt(v_hat) + eps)
        history.append(binary_cross_entropy(X, y, w))
    return w, history

runs = {'Adam': train(X_train,y_train)}

## 3. Evaluate on held-out test data and visualize

In [ ]:
results = []
for name, (weights, losses) in runs.items():
    train_metrics = evaluate(X_train, y_train, weights)
    test_metrics = evaluate(X_test, y_test, weights)
    results.append({'Optimizer': name, 'Train Loss': train_metrics['Loss'],
                    'Test Loss': test_metrics['Loss'],
                    'Accuracy': test_metrics['Accuracy'],
                    'Precision': test_metrics['Precision'],
                    'Recall': test_metrics['Recall'], 'F1': test_metrics['F1']})
results_df = pd.DataFrame(results)
display(results_df.round(4))

plt.figure(figsize=(8,4))
for name, (_, losses) in runs.items():
    plt.plot(losses, label=name)
plt.xlabel('Epoch'); plt.ylabel('Training Binary Cross-Entropy')
plt.title('Training loss comparison'); plt.legend(); plt.grid(alpha=.3)
plt.show()

for name, (weights, _) in runs.items():
    predictions = (predict_proba(X_test, weights) >= 0.5).astype(int)
    ConfusionMatrixDisplay.from_predictions(y_test, predictions,
        display_labels=['Malignant (0)', 'Benign (1)'], cmap='Blues')
    plt.title(name + ' — Test Confusion Matrix')
    plt.show()
